In [1]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

shortlist_pruned = pd.read_csv(os.path.join(out_dir, "checkpoint10_shortlist_ld_pruned.csv"))
pruned_probe_ids = shortlist_pruned["probe_id"].tolist()

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))
pruned_rows = encoded_df[encoded_df["probe_id"].isin(set(pruned_probe_ids))].copy()
pruned_rows = pruned_rows.set_index("probe_id").reindex(pruned_probe_ids)

sample_ids = pruned_rows.columns.tolist()
X_pc_input = pruned_rows.to_numpy(dtype=np.float64).T

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
status_map = meta_df.set_index("sample_id")["smoking_status"]
Y = np.array([1.0 if status_map.get(sid) == "Smoker" else 0.0 for sid in sample_ids])

X_pc_full = np.hstack([X_pc_input, Y.reshape(-1, 1)])
col_names = pruned_probe_ids + ["smoking_status"]

print("Shape:", X_pc_full.shape)
print("dtype:", X_pc_full.dtype)

Shape: (3348, 97)
dtype: float64


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
import time

start = time.time()
cg = pc(
    data=X_pc_full,
    alpha=0.05,
    indep_test=fisherz,
    stable=True,
    uc_rule=0,
    uc_priority=2,
    mvpc=False,
    depth=4,
    verbose=False
)
elapsed = time.time() - start
print(f"Completed in {elapsed:.1f} seconds")
print("Nodes:", len(cg.G.nodes))

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Depth=5, working on node 96: 100%|██████████| 97/97 [00:23<00:00, 24.55it/s]  

In [2]:
from causallearn.search.ConstraintBased.PC import pc
import inspect
print(inspect.signature(pc))

(data: 'ndarray', alpha=0.05, indep_test='fisherz', stable: 'bool' = True, uc_rule: 'int' = 0, uc_priority: 'int' = 2, mvpc: 'bool' = False, correction_name: 'str' = 'MV_Crtn_Fisher_Z', background_knowledge: 'BackgroundKnowledge | None' = None, verbose: 'bool' = False, show_progress: 'bool' = True, node_names: 'List[str] | None' = None, **kwargs)


c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import subprocess
result = subprocess.run(
    ["pip", "show", "causal-learn"],
    capture_output=True, text=True
)
print(result.stdout)

Name: causal-learn
Version: 0.1.4.7
Summary: causal-learn Python Package
Home-page: https://github.com/py-why/causal-learn
Author: 
Author-email: 
License: 
Location: C:\Users\user\Desktop\ai causal\.venv\Lib\site-packages
Requires: graphviz, joblib, matplotlib, momentchi2, networkx, numpy, pandas, pydot, scikit-learn, scipy, statsmodels, tqdm
Required-by: 



In [5]:
import causallearn.search.ConstraintBased.PC as pc_module
import inspect
print(inspect.getsource(pc_module))

from __future__ import annotations

import time
import warnings
from itertools import combinations, permutations
from typing import Dict, List, Tuple

import networkx as nx
import numpy as np
from numpy import ndarray

from causallearn.graph.GraphClass import CausalGraph
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.utils.cit import *
from causallearn.utils.PCUtils import Helper, Meek, SkeletonDiscovery, UCSepset
from causallearn.utils.PCUtils.BackgroundKnowledgeOrientUtils import \
    orient_by_background_knowledge


def pc(
    data: ndarray, 
    alpha=0.05, 
    indep_test=fisherz, 
    stable: bool = True, 
    uc_rule: int = 0, 
    uc_priority: int = 2,
    mvpc: bool = False, 
    correction_name: str = 'MV_Crtn_Fisher_Z',
    background_knowledge: BackgroundKnowledge | None = None, 
    verbose: bool = False, 
    show_progress: bool = True,
    node_names: List[str] | None = None,
    **kwargs
):
    if data.shape[0] < data.sh